# Lab 04-03: Find and Patch a Vulnerability With a Subagent

**Book chapter:** Chapter 04: Using and Customizing Coding Agents  
**Estimated time:** 30 minutes

## Overview

Cymbal Retail's support tools include an order lookup function that builds a SQL query
from text typed by a user. Its tests pass, yet it is vulnerable to SQL injection. In
this lab, you follow the remediation workflow from the chapter: you demonstrate the
attack as a baseline, delegate a review to a read-only security reviewer subagent, ask
the main agent to fix the flaw and prove the fix with a test, and then verify the
result and review the change yourself.

## Learning Objectives

By the end of this lab, you will be able to:

- Demonstrate a SQL injection vulnerability as a baseline
- Define a subagent with a read-only tool allowlist
- Delegate a security review and act on its findings
- Verify a fix independently and review it as a diff

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for Gemini requests. The lab creates no Google Cloud resources.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Practice | Why it matters |
|---|---|
| **Baseline** | Proves the problem exists and gives you something to measure the fix against |
| **Read-only reviewer** | A subagent with only read tools cannot change the code it reviews |
| **Test that proves the fix** | Stops the vulnerability from returning in a later change |
| **Human review of the diff** | The final check before a change is merged |

**SQL injection** happens when user input is inserted directly into a query, so that input
such as `x' OR '1'='1` changes the query's logic. Parameterized queries pass input
separately from the SQL, so it is always treated as data.

## Setup

In [ ]:
%pip install --quiet --upgrade google-genai

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The agent code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

# Part 1: Define the Coding Agent

These cells define the same agent that you built in Lab 04-01, including `run_subagent`,
which this lab uses for the security review.

### The Gemini Client and Helpers

This cell creates the Gemini client, defines the base instruction that every agent
run starts from, and defines `run`, a helper that runs a command and returns the end of
its output. The project and model come from the environment variables set during setup.

In [ ]:
import json
import os
import pathlib
import re
import subprocess
import sys

from google import genai
from google.genai import types

PROJECT = os.environ["GOOGLE_CLOUD_PROJECT"]
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
client = genai.Client(vertexai=True, project=PROJECT, location="global")

BASE_INSTRUCTION = """You are a coding agent working in a small Python
project. Use the tools to inspect files, make changes, and run the
tests. Make the smallest change that completes the task, run the tests
before you finish, and end with a short summary of what you changed."""


def run(command, cwd, shell=False):
    """Runs a command and returns the end of its combined output."""
    done = subprocess.run(command, cwd=cwd, shell=shell, text=True,
                          capture_output=True, timeout=120)
    return (done.stdout + done.stderr)[-3000:] or "(no output)"

### The Tools

Each tool is an ordinary Python function. The Gen AI SDK turns the function's name,
type hints, and docstring into a **tool declaration**, so the docstring is what the model
reads when it decides which tool to call.

The `inside` function resolves every path and refuses any path outside the workspace.
This is a simple **sandbox boundary**: whatever the model asks for, the file tools cannot
read or write anything else on your computer.

In [ ]:
def make_tools(root, skills):
    """Creates the agent's tools. File tools stay inside the workspace."""

    def inside(path):
        full = (root / path).resolve()
        if not full.is_relative_to(root):
            raise PermissionError(f"{path} is outside the workspace")
        return full

    def list_files() -> list[str]:
        """Lists the files in the workspace."""
        return sorted(p.relative_to(root).as_posix()
                      for p in root.rglob("*")
                      if p.is_file() and "__pycache__" not in p.parts)

    def read_file(path: str) -> str:
        """Returns the text of a workspace file.

        Args:
            path: The file path, relative to the workspace root.
        """
        return inside(path).read_text()

    def write_file(path: str, content: str) -> str:
        """Creates or replaces a workspace file.

        Args:
            path: The file path, relative to the workspace root.
            content: The complete new content of the file.
        """
        target = inside(path)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content)
        return f"Wrote {len(content)} characters to {path}."

    def run_tests() -> str:
        """Runs the project's unit tests and returns the report."""
        return run([sys.executable, "-m", "unittest", "discover", "-v"],
                   root)

    def run_command(command: str) -> str:
        """Runs a shell command in the workspace and returns its output.

        Args:
            command: The shell command to run.
        """
        return run(command, root, shell=True)

    def load_skill(name: str) -> str:
        """Returns the full instructions of an available skill.

        Args:
            name: The name of the skill to load.
        """
        return skills[name]["body"]

    return [list_files, read_file, write_file, run_tests, run_command,
            load_skill]

### Rules, Skills, and Hooks

These functions load the customizations from the same locations that Antigravity uses:

| Function | Customization | Location |
|---|---|---|
| `load_rules` | Rules added to every run | `AGENTS.md` |
| `load_skills` | Skills, shown by name and description until loaded | `.agents/skills/<name>/SKILL.md` |
| `pre_tool_use` | Hooks that can deny a tool call | `.agents/hooks.json` |

A hook receives the event as JSON on standard input and prints a JSON decision. Because
it is ordinary code, it enforces policy the same way every time.

In [ ]:
def front_matter(text):
    """Splits a Markdown file into frontmatter fields and a body."""
    match = re.match(r"---\n(.*?)\n---\n(.*)", text, re.S)
    if not match:
        return {}, text
    fields = dict(re.findall(r"^(\w+):\s*(.*)$", match.group(1), re.M))
    return fields, match.group(2).strip()


def load_rules(root):
    """Returns the project rules in AGENTS.md, if the file exists."""
    rules = root / "AGENTS.md"
    return rules.read_text() if rules.exists() else ""


def load_skills(root):
    """Finds the skills in .agents/skills/<name>/SKILL.md."""
    skills = {}
    for path in sorted(root.glob(".agents/skills/*/SKILL.md")):
        fields, body = front_matter(path.read_text())
        name = fields.get("name", path.parent.name)
        skills[name] = {"description": fields.get("description", ""),
                        "body": body}
    return skills


def pre_tool_use(root, tool, args):
    """Runs matching PreToolUse hooks and returns a reason if one denies."""
    config = root / ".agents" / "hooks.json"
    if not config.exists():
        return None
    event = json.dumps({"event": "PreToolUse", "tool_name": tool,
                        "tool_input": args})
    for group in json.loads(config.read_text()).values():
        for entry in group.get("PreToolUse", []):
            if not re.fullmatch(entry.get("matcher", ".*"), tool):
                continue
            for hook in entry["hooks"]:
                # Run Python hooks with the same interpreter as the agent.
                command = hook["command"]
                if command.startswith("python "):
                    command = f'"{sys.executable}" {command[7:]}'
                done = subprocess.run(
                    command, cwd=root, shell=True, input=event, text=True,
                    capture_output=True, timeout=hook.get("timeout", 30))
                decision = json.loads(done.stdout or "{}")
                if decision.get("decision") == "deny":
                    return decision.get("reason", "Denied by a hook.")
    return None

### The Reason-Act-Observe Loop

`run_agent` is the heart of a coding agent:

1. It builds the instruction from the base instruction, the rules, and the skill catalog.
2. It sends the conversation to Gemini, which either replies with text or asks for tools.
3. For each tool call, it runs the PreToolUse hooks, checks the tool allowlist, runs the
   tool, and records the result.
4. It sends the results back and repeats until the model replies with text or the step
   limit is reached.

Automatic function calling is turned off so that every step is visible in the activity log.

In [ ]:
def brief(value, width=72):
    """Shortens a value to one line for the activity log."""
    text = value if isinstance(value, str) else json.dumps(value)
    text = " ".join(text.split())
    return text if len(text) <= width else text[:width - 3] + "..."


def run_agent(task, workspace, system=None, allowed_tools=None,
              max_steps=20):
    """Runs the reason-act-observe loop until the model stops calling."""
    root = pathlib.Path(workspace).resolve()
    skills = load_skills(root)
    tools = {tool.__name__: tool for tool in make_tools(root, skills)}
    if allowed_tools is not None:
        tools = {n: t for n, t in tools.items() if n in allowed_tools}

    instruction = system or BASE_INSTRUCTION
    rules = load_rules(root)
    if rules:
        instruction += "\n\nProject rules from AGENTS.md:\n" + rules
    if skills and "load_skill" in tools:
        catalog = "\n".join(f"- {name}: {skill['description']}"
                            for name, skill in skills.items())
        instruction += ("\n\nAvailable skills. Call load_skill before "
                        "starting a task that matches one:\n" + catalog)

    config = types.GenerateContentConfig(
        system_instruction=instruction, tools=list(tools.values()),
        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True))
    contents = [types.Content(role="user",
                              parts=[types.Part.from_text(text=task)])]

    for step in range(1, max_steps + 1):
        response = client.models.generate_content(
            model=MODEL, contents=contents, config=config)
        contents.append(response.candidates[0].content)
        if not response.function_calls:
            print(f"\nAgent: {response.text}")
            return response.text

        replies = []
        for call in response.function_calls:
            args = dict(call.args or {})
            reason = pre_tool_use(root, call.name, args)
            if reason:
                result = {"error": f"Blocked by a hook: {reason}"}
            elif call.name not in tools:
                result = {"error": f"The tool {call.name} is not allowed."}
            else:
                try:
                    result = {"result": tools[call.name](**args)}
                except Exception as error:  # Report errors to the model.
                    result = {"error": str(error)}
            print(f"[{step}] {call.name}({brief(args, 40)}) -> "
                  f"{brief(result)}")
            replies.append(types.Part.from_function_response(
                name=call.name, response=result))
        contents.append(types.Content(role="user", parts=replies))

    print("Stopped: the step limit was reached.")
    return None

### Subagents

A subagent is defined in a Markdown file. Its frontmatter `tools` list becomes a **tool
allowlist**, and its body becomes the system instruction. The subagent starts with a fresh
conversation, so it does not inherit the main agent's history.

In [ ]:
def run_subagent(name, task, workspace):
    """Runs a subagent defined in .agents/agents/<name>.md."""
    root = pathlib.Path(workspace).resolve()
    path = root / ".agents" / "agents" / f"{name}.md"
    fields, body = front_matter(path.read_text())
    tools = re.findall(r"\w+", fields.get("tools", ""))
    print(f"Delegating to the {name} subagent with tools: {tools}")
    return run_agent(task, root, system=body, allowed_tools=tools)

# Part 2: Run the Remediation Workflow

## Step 1: Create an order lookup service that has a security flaw

The `find_orders` function builds its SQL query by inserting the customer text into an f-string. Its two tests pass, but neither tries an attack.

In [ ]:
import difflib
import importlib.util
import pathlib
import sys
import textwrap


WORKSPACE = pathlib.Path("workspace-orders").resolve()
WORKSPACE.mkdir(exist_ok=True)

(WORKSPACE / "orders.py").write_text(textwrap.dedent('''\
    """Order lookups for Cymbal Retail support tools."""
    import sqlite3


    def connect():
        """Returns an in-memory database with sample orders."""
        db = sqlite3.connect(":memory:")
        db.execute("CREATE TABLE orders (id TEXT, customer TEXT, "
                   "status TEXT)")
        db.executemany("INSERT INTO orders VALUES (?, ?, ?)", [
            ("10421", "alice@example.com", "shipped"),
            ("10422", "bob@example.com", "processing"),
            ("10423", "carol@example.com", "delivered")])
        return db


    def find_orders(db, customer):
        """Returns the orders that belong to one customer."""
        query = ("SELECT id, status FROM orders "
                 f"WHERE customer = '{customer}'")
        return db.execute(query).fetchall()
    '''))

(WORKSPACE / "test_orders.py").write_text(textwrap.dedent('''\
    import unittest

    from orders import connect, find_orders


    class FindOrdersTest(unittest.TestCase):
        def setUp(self):
            self.db = connect()

        def test_returns_only_the_customers_orders(self):
            self.assertEqual(find_orders(self.db, "bob@example.com"),
                             [("10422", "processing")])

        def test_unknown_customer_has_no_orders(self):
            self.assertEqual(find_orders(self.db, "x@example.com"), [])
    '''))
ORIGINAL = (WORKSPACE / "orders.py").read_text()

## Step 2: Show the attack before fixing anything

Establish a baseline before changing anything. `try_attack` loads the current `orders.py` and looks up orders for the classic injection string `x' OR '1'='1`.

In [ ]:
ATTACK = "x' OR '1'='1"


def try_attack():
    """Loads the current orders.py and looks up orders for ATTACK."""
    spec = importlib.util.spec_from_file_location(
        "orders", WORKSPACE / "orders.py")
    orders = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(orders)
    return orders.find_orders(orders.connect(), ATTACK)


print("Orders returned for the attack:", try_attack())
print(run([sys.executable, "-m", "unittest", "discover"], WORKSPACE))

**Expected result:** the attack returns all three orders, even though both tests pass.

## Step 3: Define a read-only security reviewer subagent and run it

The reviewer is a **subagent** with a read-only tool allowlist. Keeping reviewers read-only means a review can never change the code it is reviewing.

In [ ]:
reviewer = WORKSPACE / ".agents" / "agents" / "security-reviewer.md"
reviewer.parent.mkdir(parents=True, exist_ok=True)
reviewer.write_text(textwrap.dedent("""\
    ---
    name: security-reviewer
    description: Reviews Python code for security flaws before merging.
    tools: [list_files, read_file]
    ---
    You are a security reviewer for Cymbal Retail. Read the code and
    report each security flaw with the file, the function, the risk,
    and a suggested fix. You have read-only tools; never change files.
    """))
report = run_subagent("security-reviewer",
                      "Review this project for security flaws.", WORKSPACE)

**Expected result:** the subagent runs with `['list_files', 'read_file']` and reports a SQL injection flaw in `find_orders`. If it tries to write a file, the call is refused.

## Step 4: Ask the main agent to fix the flaw and prove the fix

The main agent receives the review and must fix the flaw without changing normal behavior, and prove the fix with a new test.

In [ ]:
run_agent("A security review found this problem:\n"
          f"{report}\n"
          "Fix the vulnerability in orders.py without changing its "
          "behavior for normal input. Add a unit test that proves the "
          "attack string x' OR '1'='1 returns no orders, and run the "
          "tests.", WORKSPACE)

**Expected result:** the agent replaces the f-string with a parameterized query, adds a test for the attack string, and runs the tests.

## Step 5: Verify the fix independently and review the change

Verify the fix yourself and review the exact change, as you would before approving a pull request.

In [ ]:
print("Orders returned for the attack:", try_attack())
print(run([sys.executable, "-m", "unittest", "discover", "-v"], WORKSPACE))
FIXED = (WORKSPACE / "orders.py").read_text()
print("".join(difflib.unified_diff(
    ORIGINAL.splitlines(keepends=True), FIXED.splitlines(keepends=True),
    "orders.py (before)", "orders.py (after)")))

**Expected result:** the attack returns `[]`, all three tests pass, and the diff shows only the query change.

## Check Your Understanding

**1. Why did the vulnerable code pass its tests?**

<details><summary>Show answer</summary>

The tests covered only normal input. No test tried an attack string, so a baseline that demonstrates the attack was needed.

</details>

**2. What prevents the security reviewer from changing the code?**

<details><summary>Show answer</summary>

Its definition allows only list_files and read_file, so the harness refuses any other tool call.

</details>

**3. Why is a parameterized query safe when an f-string is not?**

<details><summary>Show answer</summary>

A parameterized query sends the input separately from the SQL text, so the database treats it as data and it cannot change the query's logic.

</details>

## Clean Up

Run the next cell to delete the two workspaces created by the Chapter 04 labs. The labs
create no Google Cloud resources.

In [ ]:
import pathlib
import shutil

for name in ["workspace-pricing", "workspace-orders"]:
    folder = pathlib.Path(name)
    if folder.exists():
        shutil.rmtree(folder)
        print("Deleted", folder.resolve())
print("The Chapter 04 labs create no Google Cloud resources.")

## Summary

You followed a complete remediation workflow: demonstrate the vulnerability, review with
a read-only subagent, fix with a test that proves the fix, and verify and review the diff
yourself. This is the pattern the exam rewards for using coding agents on security fixes.